# Day 2 Session 1: Simulating Dynamical Systems


> **Participant notebook:** Work through the examples in order. For exercises that ask you to modify an existing example, a copied workspace or starter cell is included so you can change the named values without editing the original demonstration cell.


### <center> Simulating Dynamical Systems </center>

#### Start with continuous-time dynamical systems

In many mathematics courses, a **dynamical system** first appears as a differential equation. The state of the system changes continuously in time, and the rate of change is described by an equation.

A general one-dimensional continuous-time dynamical system can be written as

$$
\begin{aligned}
\frac{dX}{dt} &= f(X(t), t),\\
X(0) &= X_0.
\end{aligned}
$$

Here:

- $X(t)$ is the state of the system at time $t$.
- $f(X(t),t)$ tells us the rate of change at that moment.
- $X_0$ is the initial condition.

The key modeling question is:

> If we know the current state and the rate of change, how can we predict the future state?

For a computer, the challenge is that it cannot move through every continuous instant of time. We have to choose time points

$$
t_0, t_1, t_2, \ldots
$$

and approximate the continuous solution on that grid. This is where **discretization** enters.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### A first continuous example: exponential growth

Suppose a bacteria population grows at a rate proportional to the current population:

$$
\frac{dX}{dt} = rX(t), \qquad X(0)=X_0.
$$

The parameter $r$ is the growth rate. This differential equation has the exact solution

$$
X(t)=X_0 e^{rt}.
$$

If we want the population to double every hour, then

$$
X(1)=2X_0.
$$

Substituting into the solution gives

$$
X_0 e^r = 2X_0
\quad \Longrightarrow \quad
r=\ln(2).
$$

So the continuous model

$$
\frac{dX}{dt}=\ln(2)X(t)
$$

matches the idea of hourly doubling when we look at integer-hour time points.


In [ ]:
X0 = 1
r = np.log(2)

t_cont = np.linspace(0, 10, 300)
X_exact = X0 * np.exp(r * t_cont)

t_hourly = np.arange(0, 10 + 1, 1)
X_hourly = X0 * 2 ** t_hourly

plt.figure(figsize=(8, 4))
plt.plot(t_cont, X_exact, label="Continuous solution: $X(t)=e^{rt}$")
plt.scatter(t_hourly, X_hourly, label="Hourly doubling points", zorder=3)
plt.title("Continuous Growth Model with Hourly Doubling")
plt.xlabel("Time (hours)")
plt.ylabel("Population")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Euler's Method: discretizing a differential equation

Euler's method is one of the simplest ways to approximate a continuous-time dynamical system with a computer.

Start with the differential equation

$$
\frac{dX}{dt}=f(X(t),t).
$$

The derivative can be approximated by a finite difference:

$$
\frac{dX}{dt}
\approx
\frac{X(t+\Delta t)-X(t)}{\Delta t}.
$$

Substitute this approximation into the differential equation:

$$
\frac{X(t+\Delta t)-X(t)}{\Delta t}
\approx
f(X(t),t).
$$

Solving for the next state gives the **Euler update rule**:

$$
X(t+\Delta t)
\approx
X(t)+\Delta t\, f(X(t),t).
$$

Using grid notation $t_n=n\Delta t$ and $X_n \approx X(t_n)$, we write

$$
\boxed{
X_{n+1}=X_n+\Delta t\,f(X_n,t_n)
}
$$

This is the bridge from differential equations to code:

1. Choose a step size $\Delta t$.
2. Start from the initial condition $X_0$.
3. Repeatedly update the state using the slope at the current point.
4. Plot the approximate solution.


In [ ]:
def euler_1d(f, X0, t_grid):
    """Euler's method for a one-dimensional differential equation dX/dt = f(X, t)."""
    X = np.zeros(len(t_grid))
    X[0] = X0
    delta_t = t_grid[1] - t_grid[0]

    for n in range(1, len(t_grid)):
        X_prev = X[n - 1]
        t_prev = t_grid[n - 1]
        X[n] = X_prev + delta_t * f(X_prev, t_prev)

    return X


def exponential_rhs(X, t):
    return r * X


plt.figure(figsize=(8, 4))
plt.plot(t_cont, X_exact, label="Exact continuous solution", linewidth=3)

for delta_t in [1.0, 0.5, 0.1]:
    t_grid = np.arange(0, 10 + delta_t, delta_t)
    X_euler = euler_1d(exponential_rhs, X0, t_grid)
    plt.plot(t_grid, X_euler, marker="o", markersize=3, label=f"Euler, Δt={delta_t}")

plt.title("Euler Approximation of Exponential Growth")
plt.xlabel("Time (hours)")
plt.ylabel("Population")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### One-dimensional case: logistic growth

The exponential model assumes unlimited resources. In practice, growth slows down as the population approaches a carrying capacity.

A common continuous-time model is the **logistic growth equation**:

$$
\frac{dX}{dt}=rX(t)\left(1-\frac{X(t)}{N}\right).
$$

Here:

- $r$ is the growth rate.
- $N$ is the carrying capacity.
- The factor $\left(1-\frac{X(t)}{N}\right)$ slows growth when $X(t)$ is close to $N$.

We can approximate this model with Euler's method:

$$
X_{n+1}
=
X_n+\Delta t\,rX_n\left(1-\frac{X_n}{N}\right).
$$


In [ ]:
r = 1.0
N = 100
X0 = 1
delta_t = 0.1
t_grid = np.arange(0, 15 + delta_t, delta_t)

def logistic_rhs(X, t):
    return r * X * (1 - X / N)

X_logistic = euler_1d(logistic_rhs, X0, t_grid)

plt.figure(figsize=(8, 4))
plt.plot(t_grid, X_logistic, label="Euler approximation")
plt.axhline(N, linestyle="--", label="Carrying capacity")
plt.title("One-Dimensional Logistic Growth")
plt.xlabel("Time")
plt.ylabel("Population")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 1: Euler's Method and Step Size

**Core:** Change `delta_t` to `1.0`, `0.5`, and `0.05`. What changes in the graph?

**Stretch:** Try two different values of the carrying capacity `N`. How does the plateau change?

**Classroom connection:** Ask students to explain why the same differential equation can produce different-looking computer graphs when the step size changes.


### Two-dimensional dynamical systems

Dynamical systems do not have to be limited to one variable. For a two-dimensional system, the state is a vector:

$$
X(t)=
\begin{bmatrix}
x(t)\\
y(t)
\end{bmatrix}.
$$

Euler's method has the same structure:

$$
X_{n+1}=X_n+\Delta t\, f(X_n,t_n),
$$

but now $X_n$ and $f(X_n,t_n)$ are vectors.

To illustrate this, we will use the Lotka-Volterra predator-prey equations:

$$
\begin{aligned}
\frac{dx}{dt} &= \alpha x-\beta xy,\\
\frac{dy}{dt} &= -\gamma y+\delta xy.
\end{aligned}
$$

Here $x(t)$ is the prey population and $y(t)$ is the predator population.


In [ ]:
par = np.array([1.1, 0.4, 0.4, 0.1])  # alpha, beta, gamma, delta

delta_t = 0.001
t_grid = np.arange(0, 30 + delta_t, delta_t)

X = np.zeros((len(t_grid), 2))
X[0, :] = np.array([10.0, 1.0])  # initial condition: prey, predator

def lotka_volterra_rhs(X, params):
    prey, predator = X
    alpha, beta, gamma, delta = params
    d_prey = alpha * prey - beta * prey * predator
    d_predator = -gamma * predator + delta * prey * predator
    return np.array([d_prey, d_predator])

for n in range(1, len(t_grid)):
    X_prev = X[n - 1]
    X[n] = X_prev + delta_t * lotka_volterra_rhs(X_prev, par)

plt.figure(figsize=(8, 4))
plt.plot(t_grid, X, label=["Prey", "Predator"])
plt.title("Two-Dimensional Lotka-Volterra System")
plt.xlabel("Time")
plt.ylabel("Population")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 2: Two-Dimensional Dynamics

---

(a) What would the solutions of $x(t)$ and $y(t)$ look like if the $xy$ interaction terms were omitted?

(b) What happens if you set $\Delta t$ to `0.1` or `1.0`? How does the numerical approximation change?

(c) Plot $x(t)$ against $y(t)$. What can we infer from the resulting phase plot?


#### Exercise 2 Workspace and Visualization Support

Use this cell if participants need a starting point for the Lotka-Volterra exercise. It creates the phase plot requested in part (c).

**Core:** run the code and describe the cycle.

**Stretch:** change one parameter and explain the biological interpretation.


In [ ]:
# Exercise 2 workspace: phase plot for predator-prey dynamics
plt.figure(figsize=(6, 5))
plt.plot(X[:, 0], X[:, 1])
plt.xlabel("Prey")
plt.ylabel("Predator")
plt.title("Phase Plot: Predator vs. Prey")
plt.grid(True, alpha=0.3)
plt.show()